# Rotated Viewports

This tutorial demonstrates viewport rotation in `grid_py`, following the
R vignette *Rotated Viewports*.

Any `Viewport` can be given an `angle` parameter (in degrees) so that
all drawing inside it is rotated around the viewport centre.

In [ ]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np

import grid_py as gp

## A simple rotated multipanel plot

We push a viewport rotated by 15 degrees and draw a multipanel demo
inside it.  Everything drawn within the viewport is rotated as a unit.

In [ ]:
gp.grid_newpage()

gp.push_viewport(gp.Viewport(width=0.8, height=0.8, angle=15))
gp.grid_multipanel(newpage=False)
gp.pop_viewport()

plt.gcf()

## Rotated text at various angles

Text inside a rotated viewport combines the viewport rotation with
any per-grob `rot` parameter.

In [ ]:
gp.grid_newpage()

angles = [0, 30, 45, 60, 90]
n = len(angles)

for i, ang in enumerate(angles):
    vp = gp.Viewport(
        x=(i + 0.5) / n,
        y=0.5,
        width=1.0 / n,
        height=0.8,
        angle=ang,
    )
    gp.push_viewport(vp)
    gp.grid_rect(gp=gp.Gpar(col="grey", lty="dashed"))
    gp.grid_text(
        f"{ang} deg", 0.5, 0.5,
        gp=gp.Gpar(fontsize=10),
    )
    gp.pop_viewport()

plt.gcf()

## Scatterplot with a rotated boxplot overlay

A more involved example: we draw a scatterplot of correlated data and
overlay a rotated viewport at 45 degrees to show a boxplot of the
residuals from the y=x line.

In [ ]:
rng = np.random.default_rng(42)
x = rng.standard_normal(50)
y = x + rng.normal(1, 2, size=50)

all_vals = np.concatenate([x, y])
lo, hi = all_vals.min() - 0.5, all_vals.max() + 0.5

gp.grid_newpage()

# Main viewport with axes
vp_main = gp.Viewport(
    width=gp.Unit(4, "inches"),
    height=gp.Unit(4, "inches"),
    xscale=[lo, hi],
    yscale=[lo, hi],
)
gp.push_viewport(vp_main)
gp.grid_rect()
gp.grid_xaxis()
gp.grid_text("Test", y=gp.Unit(-3, "lines"))
gp.grid_yaxis()
gp.grid_text("Retest", x=gp.Unit(-3, "lines"), rot=90)

# Points in a nested viewport with tighter scale
scale = [lo + 0.5, hi - 0.5]
inner_lay = gp.GridLayout(
    nrow=2, ncol=2,
    widths=gp.Unit([3, 1], "inches"),
    heights=gp.Unit([1, 3], "inches"),
)
gp.push_viewport(gp.Viewport(layout=inner_lay))
gp.push_viewport(gp.Viewport(
    layout_pos_row=2, layout_pos_col=1,
    xscale=scale, yscale=scale,
))
gp.grid_lines()  # diagonal reference line
gp.grid_points(
    gp.Unit(x.tolist(), "native"),
    gp.Unit(y.tolist(), "native"),
    gp=gp.Gpar(col="blue"),
)
gp.pop_viewport()

# Rotated viewport for the boxplot-style indicator
diffs = y - x
d_range = diffs.max() - diffs.min()
vp_rot = gp.Viewport(
    x=gp.Unit(3, "inches"),
    y=gp.Unit(3, "inches"),
    width=gp.Unit(0.5, "inches"),
    height=gp.Unit(d_range * np.sin(np.pi / 4) / (scale[1] - scale[0]) * 3, "inches"),
    just=["centre", "centre"],
    angle=45,
    gp=gp.Gpar(col="red"),
    yscale=[-d_range / 2, d_range / 2],
)
gp.push_viewport(vp_rot)

# Draw a simple median line in the rotated viewport
med = float(np.median(diffs))
q1, q3 = float(np.percentile(diffs, 25)), float(np.percentile(diffs, 75))
gp.grid_rect(
    x=0.5,
    y=gp.Unit(q1, "native"),
    width=0.6,
    height=gp.Unit(q3 - q1, "native"),
    just=["centre", "bottom"],
    gp=gp.Gpar(fill="orange"),
)
gp.grid_segments(
    0.2, gp.Unit(med, "native"),
    0.8, gp.Unit(med, "native"),
    gp=gp.Gpar(lwd=2),
)
gp.grid_yaxis(main=False)
gp.pop_viewport(3)

plt.gcf()

The orange box and red axis are drawn inside a viewport rotated 45
degrees, aligned with the y=x diagonal of the scatterplot.